# Tax AI V1.2d — Colab GPU + Gemma 4 E4B + api-ocr-2025

架構：**GitHub Pages → Vercel 免費 HTTPS Proxy → Colab GPU → api-ocr-2025 → Gemma 4 E4B**

V1.2d 修正：
- 移除本流程不使用、但會限制 NumPy 版本的 `numba/librosa`，解決 Colab 內建 Numba 與 api-ocr-2025 新版 NumPy 的衝突。
- 固定 `requests==2.32.4`，符合 Google Colab 目前相依。
- Gemma VLM sidecar 與模型在同一個 Notebook process 執行。
- 買受人8格=`buyer_tax_id`；右下統一發票專用章=`seller_tax_id`；檢查碼只驗證、不改值。

**請使用全新的 Colab Runtime 執行本 Notebook。**


In [ ]:
import sys, subprocess, torch
print('Python:', sys.version)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
subprocess.run(['nvidia-smi'], check=False)
if not torch.cuda.is_available():
    raise RuntimeError('請先在 Colab：執行階段 → 變更執行階段類型 → GPU')

## 1. 清理 Colab 不需要的相依套件，再安裝核心環境
本專案只做影像發票辨識，不需要 librosa/numba 的音訊與 JIT 路徑。

In [ ]:
!apt-get -qq update
!apt-get -qq install -y libzbar0
!pip -q uninstall -y librosa numba || true
!pip -q install -U "requests==2.32.4" "transformers>=5.5.0" accelerate bitsandbytes huggingface_hub fastapi uvicorn python-multipart pillow
print('✅ Base environment installed')

## 2. 下載並固定 api-ocr-2025


In [ ]:
import os, shutil, subprocess
REPO='/content/api-ocr-2025'
if os.path.exists(REPO): shutil.rmtree(REPO)
subprocess.run(['git','clone','https://github.com/adi-gov-tw/api-ocr-2025.git',REPO],check=True)
subprocess.run(['git','checkout','5ef5794c1b0c3fc640d6ac8c8d26562b6c035202'],cwd=REPO,check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',REPO+'/requirements.txt'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','requests==2.32.4'],check=True)
print('✅ api-ocr-2025 installed')

In [ ]:
import numpy, requests
print('NumPy:', numpy.__version__)
print('requests:', requests.__version__)
try:
    import numba
    print('⚠️ numba still present:', numba.__version__)
except Exception:
    print('✅ numba not installed — expected for this image-only pipeline')

## 3. Hugging Face 登入（Gemma 若要求授權時執行）
若下載模型時出現 401/403，先到 Hugging Face 接受 Gemma 模型條款，再執行本格。

In [ ]:
from huggingface_hub import login
# 若模型可直接下載，可先跳過本格；遇到 401/403 再回來執行 login()
# login()

## 4. 載入 Gemma 4 E4B（4-bit 優先）


In [ ]:
import gc, torch
from transformers import AutoProcessor, AutoModelForMultimodalLM, BitsAndBytesConfig
MODEL_ID='google/gemma-4-E4B-it'
processor=AutoProcessor.from_pretrained(MODEL_ID)
qconfig=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',bnb_4bit_compute_dtype=torch.float16,bnb_4bit_use_double_quant=True)
try:
    model=AutoModelForMultimodalLM.from_pretrained(MODEL_ID,quantization_config=qconfig,device_map='auto',dtype=torch.float16)
    print('✅ Gemma 4 E4B loaded in 4-bit')
except Exception as e:
    print('⚠️ 4-bit load failed:', repr(e))
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    model=AutoModelForMultimodalLM.from_pretrained(MODEL_ID,device_map='auto',dtype='auto')
    print('✅ Gemma 4 E4B loaded without 4-bit')
print('Model device:', next(model.parameters()).device)

## 5. 建立 OpenAI-compatible Gemma Vision sidecar (:8001)


In [ ]:
import base64, io, time, threading, requests
from fastapi import FastAPI, Request
from PIL import Image
import uvicorn
vlm_app=FastAPI(title='Gemma 4 E4B sidecar')
def decode_data_image(value):
    if isinstance(value,dict): value=value.get('url')
    if isinstance(value,str) and value.startswith('data:') and ',' in value:
        return Image.open(io.BytesIO(base64.b64decode(value.split(',',1)[1]))).convert('RGB')
    return None
def collect_parts(messages):
    images=[]; texts=[]
    for msg in messages or []:
        content=msg.get('content','')
        if isinstance(content,str): texts.append(content)
        elif isinstance(content,list):
            for part in content:
                typ=part.get('type')
                if typ in ('text','input_text'): texts.append(part.get('text',''))
                elif typ in ('image_url','input_image'):
                    img=decode_data_image(part.get('image_url') or part.get('image'))
                    if img is not None: images.append(img)
    return images,'\n'.join(texts)
@vlm_app.get('/health')
async def health(): return {'status':'ok','model':MODEL_ID,'cuda':torch.cuda.is_available()}
@vlm_app.get('/v1/models')
async def models(): return {'object':'list','data':[{'id':MODEL_ID,'object':'model'}]}
@vlm_app.post('/v1/chat/completions')
async def chat(req:Request):
    body=await req.json(); images,text=collect_parts(body.get('messages',[]))
    guard=('You are a Taiwan unified-invoice vision extraction model. For triplicate/manual invoices, buyer_tax_id is exactly the eight boxes immediately after or below 買受人/統一編號 in the upper-left buyer block. seller_tax_id is the tax ID inside the lower-right 統一發票專用章. Never swap them. Read all 8 buyer boxes left to right. Never use checksum to invent, rescue, substitute, or correct digits. If unreadable return null. Return only requested JSON/data.')
    if images: messages=[{'role':'user','content':[{'type':'image','image':images[0]},{'type':'text','text':guard+'\n'+text}]}]
    else: messages=[{'role':'user','content':guard+'\n'+(text or 'Reply OK')}]
    inputs=processor.apply_chat_template(messages,tokenize=True,return_dict=True,return_tensors='pt',add_generation_prompt=True,enable_thinking=False)
    inputs={k:(v.to(model.device) if hasattr(v,'to') else v) for k,v in inputs.items()}
    n=inputs['input_ids'].shape[-1]
    with torch.inference_mode(): out=model.generate(**inputs,max_new_tokens=min(int(body.get('max_tokens') or 900),1200),do_sample=False)
    raw=processor.decode(out[0][n:],skip_special_tokens=True).strip()
    return {'id':'chatcmpl-gemma4e4b','object':'chat.completion','created':int(time.time()),'model':MODEL_ID,'choices':[{'index':0,'message':{'role':'assistant','content':raw},'finish_reason':'stop'}]}
def run_vlm(): uvicorn.Server(uvicorn.Config(vlm_app,host='127.0.0.1',port=8001,log_level='warning')).run()
threading.Thread(target=run_vlm,daemon=True).start()
for _ in range(90):
    try:
        r=requests.get('http://127.0.0.1:8001/health',timeout=2)
        if r.ok: print('✅ VLM sidecar:',r.json()); break
    except Exception: time.sleep(1)
else: raise RuntimeError('Gemma sidecar failed to start')

## 6. 啟動 api-ocr-2025 (:8080)


In [ ]:
import os, subprocess, time, requests
env=os.environ.copy()
env.update({'APIOCR_VLM_BACKEND':'local','APIOCR_LOCAL_VLM_ENABLED':'true','APIOCR_LOCAL_VLM_URL':'http://127.0.0.1:8001/v1','APIOCR_LOCAL_VLM_MODEL':MODEL_ID,'APIOCR_LOCAL_VLM_TIMEOUT':'180','APIOCR_VLM_ENABLED':'true','APIOCR_USE_GPU':'false','APIOCR_DESKEW':'true','APIOCR_UPSCALE_MIN_SIDE':'1400','APIOCR_MIN_CONFIDENCE':'0.20'})
api_proc=subprocess.Popen([sys.executable,'-m','uvicorn','app.main:app','--host','127.0.0.1','--port','8080','--workers','1'],cwd=REPO,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
for _ in range(120):
    try:
        r=requests.get('http://127.0.0.1:8080/health',timeout=2)
        if r.ok: print('✅ api-ocr-2025:',r.json()); break
    except Exception: time.sleep(1)
else:
    try: print(api_proc.stdout.read(8000))
    except Exception: pass
    raise RuntimeError('api-ocr-2025 failed to start')

## 7. 建立免費 Cloudflare HTTPS Tunnel


In [ ]:
import os, subprocess, re, time, requests
cf='/content/cloudflared'
if not os.path.exists(cf):
    subprocess.run(['wget','-q','https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64','-O',cf],check=True); os.chmod(cf,0o755)
tunnel_proc=subprocess.Popen([cf,'tunnel','--url','http://127.0.0.1:8080','--no-autoupdate'],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
public_url=None; deadline=time.time()+60
while time.time()<deadline:
    line=tunnel_proc.stdout.readline()
    if line:
        print(line.rstrip())
        m=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',line)
        if m: public_url=m.group(0); break
if not public_url: raise RuntimeError('Tunnel URL not found')
print('\nCOLAB_BACKEND_URL =',public_url)
print(requests.get(public_url+'/health',timeout=30).json())

## 8. 可選：直接在 Colab 上傳一張發票測試


In [ ]:
from google.colab import files
import requests, json
uploaded=files.upload()
for name,data in uploaded.items():
    r=requests.post(public_url+'/v1/invoice',files={'file':(name,data,'image/jpeg')},data={'engine':'vlm','slim':'false','include_image':'false'},timeout=240)
    print('HTTP',r.status_code)
    try: print(json.dumps(r.json(),ensure_ascii=False,indent=2))
    except Exception: print(r.text[:4000])